# Notebook 9: Recurrent Neural Networks (RNNs)
**Estimated time:** 35-45 min
**Prerequisites:** Notebooks 1-8

---
## What you will learn
1. Why standard networks fail on sequential data
2. How an RNN cell works (step by step)
3. Using `nn.RNN` in PyTorch
4. The vanishing gradient problem in RNNs
5. Building a character-level sequence model
6. Working with variable-length sequences

## 1. Sequential Data — Why Order Matters

**Imagine reading this sentence with the words scrambled:**
`"bank the money the deposited man"` vs `"the man deposited the money in the bank"`

The words are the same. The order gives them meaning.

CNNs and MLPs see each input independently — they have no memory of what came before.
**RNNs** maintain a **hidden state** — a memory that carries information from previous steps.

Examples of sequential data:
- **Text**: each word/character depends on what came before
- **Time series**: stock prices, sensor readings, weather
- **Audio**: each sound frame depends on the previous ones
- **Video**: each frame follows the previous one

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

## 2. The RNN Cell — Step by Step

**Imagine you're reading a book and you have a notepad.**
After each sentence, you update your notepad with key information.
When you read the next sentence, you use both the sentence AND your current notepad.
The notepad is the **hidden state**.

Mathematically, at each time step `t`:
```
h_t = tanh(W_hh * h_{t-1} + W_xh * x_t + b)
```

- `x_t` = input at step t
- `h_{t-1}` = hidden state from the previous step
- `h_t` = new hidden state
- `W_xh`, `W_hh` = weight matrices (shared across ALL time steps!)
- **Weight sharing** is key: the same weights process every position

In [ ]:
# Manual RNN cell to see what's happening inside
class ManualRNNCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.W_xh = nn.Linear(input_size,  hidden_size, bias=True)
        self.W_hh = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(self, x, h_prev):
        # x      : (batch, input_size)
        # h_prev : (batch, hidden_size)
        return torch.tanh(self.W_xh(x) + self.W_hh(h_prev))


# Process a sequence manually, step by step
batch_size  = 2
seq_len     = 5
input_size  = 4
hidden_size = 8

cell = ManualRNNCell(input_size, hidden_size)

# Sequence: (seq_len, batch, features)
x_seq = torch.randn(seq_len, batch_size, input_size)
h = torch.zeros(batch_size, hidden_size)    # initial hidden state

hidden_states = []
for t in range(seq_len):
    h = cell(x_seq[t], h)
    hidden_states.append(h)
    print(f't={t}: h norm = {h.norm():.4f}')

print(f'Final hidden state shape: {h.shape}')

## 3. Using `nn.RNN`

PyTorch's `nn.RNN` processes the entire sequence efficiently.

Key parameters:
```python
nn.RNN(
    input_size,          # size of each input token
    hidden_size,         # size of the hidden state
    num_layers=1,        # stack multiple RNN layers
    batch_first=False,   # if True: input shape is (batch, seq, features)
    dropout=0.0,         # dropout between layers (not applied to last layer)
    bidirectional=False  # if True: process sequence forward AND backward
)
```

Input/output shapes:
- Input: `(seq_len, batch, input_size)` — or `(batch, seq_len, input_size)` if `batch_first=True`
- Output: `(seq_len, batch, hidden_size)` — all hidden states
- Final `h_n`: `(num_layers, batch, hidden_size)` — last hidden state only

In [ ]:
# Using nn.RNN
rnn = nn.RNN(input_size=4, hidden_size=8, num_layers=1, batch_first=True)

x = torch.randn(2, 5, 4)   # (batch=2, seq_len=5, features=4)
h0 = torch.zeros(1, 2, 8)  # (num_layers=1, batch=2, hidden=8)

output, h_n = rnn(x, h0)

print(f'Input  shape: {x.shape}')
print(f'Output shape: {output.shape}')    # (2, 5, 8) — hidden state at every step
print(f'h_n    shape: {h_n.shape}')       # (1, 2, 8) — final hidden state only
print()
print('output[:, -1, :] is same as h_n[0]:', torch.allclose(output[:, -1, :], h_n[0]))

## 4. RNN for Sequence Classification

**Pattern 1: Many-to-one** — predict ONE label from the whole sequence
(e.g., sentiment analysis: "This is great!" → positive)

Use the **last hidden state** as the representation of the whole sequence.

In [ ]:
class RNNClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, num_layers=2, dropout=0.3):
        super().__init__()
        self.rnn = nn.RNN(
            input_size, hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        output, h_n = self.rnn(x)
        # Use the last layer's final hidden state
        last_h = h_n[-1]            # (batch, hidden_size)
        return self.classifier(last_h)


model = RNNClassifier(input_size=16, hidden_size=64, num_classes=3)
x = torch.randn(32, 20, 16)    # batch=32, seq_len=20, features=16
out = model(x)
print(f'Output shape: {out.shape}')  # (32, 3)

## 5. Character-Level Language Model

A **language model** predicts the next character given the previous ones.

**Why character level?**
- No vocabulary needed — just the 26 letters + punctuation
- Simple but powerful demonstration of sequence modeling
- Generates text character by character

In [ ]:
# Prepare character-level dataset from a small text
text = ('to be or not to be that is the question whether tis nobler in the mind '
        'to suffer the slings and arrows of outrageous fortune or to take arms '
        'against a sea of troubles and by opposing end them')

chars   = sorted(set(text))
char2id = {c: i for i, c in enumerate(chars)}
id2char = {i: c for c, i in char2id.items()}
vocab_size = len(chars)

print(f'Text length : {len(text)} chars')
print(f'Vocabulary  : {vocab_size} unique chars')
print(f'Chars: {chars}')

# Encode text
encoded = torch.tensor([char2id[c] for c in text], dtype=torch.long)

# Create (input, target) pairs with sliding window
SEQ_LEN = 20

def make_sequences(encoded, seq_len):
    inputs, targets = [], []
    for i in range(len(encoded) - seq_len):
        inputs.append(encoded[i : i + seq_len])
        targets.append(encoded[i + 1 : i + seq_len + 1])   # shifted by 1
    return torch.stack(inputs), torch.stack(targets)

inputs, targets = make_sequences(encoded, SEQ_LEN)
print(f'Dataset: {inputs.shape[0]} sequences of length {SEQ_LEN}')
print(f'Example input : {"".join(id2char[i.item()] for i in inputs[0])}')
print(f'Example target: {"".join(id2char[i.item()] for i in targets[0])}')

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim

# One-hot encode: (batch, seq_len) -> (batch, seq_len, vocab_size)
def to_onehot(x, vocab_size):
    return torch.zeros(*x.shape, vocab_size).scatter_(-1, x.unsqueeze(-1), 1.0)

dataset    = TensorDataset(inputs, targets)
loader     = DataLoader(dataset, batch_size=32, shuffle=True)


class CharRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_layers=2):
        super().__init__()
        self.rnn       = nn.RNN(vocab_size, hidden_size, num_layers=num_layers,
                                batch_first=True, dropout=0.2)
        self.fc        = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, h=None):
        out, h = self.rnn(x, h)
        logits = self.fc(out)   # (batch, seq_len, vocab_size)
        return logits, h


rnn_model = CharRNN(vocab_size=vocab_size, hidden_size=128)
optimizer = optim.Adam(rnn_model.parameters(), lr=3e-3)
criterion = nn.CrossEntropyLoss()

EPOCHS = 50
for epoch in range(EPOCHS):
    rnn_model.train()
    total_loss = 0
    for x_b, y_b in loader:
        x_oh = to_onehot(x_b, vocab_size)  # one-hot encode inputs
        optimizer.zero_grad()
        logits, _ = rnn_model(x_oh)
        # Flatten for cross entropy: (batch*seq_len, vocab) vs (batch*seq_len,)
        loss = criterion(logits.reshape(-1, vocab_size), y_b.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(rnn_model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}: loss={total_loss/len(loader):.4f}')

In [ ]:
# Generate text from the trained model
def generate(model, seed_text, n_chars=100, temperature=0.8):
    model.eval()
    generated = seed_text
    h = None

    for _ in range(n_chars):
        # Encode current sequence
        x = torch.tensor([[char2id.get(c, 0) for c in generated[-SEQ_LEN:]]], dtype=torch.long)
        x_oh = to_onehot(x, vocab_size)

        with torch.no_grad():
            logits, h = model(x_oh, h)

        # Sample from last position
        last_logits = logits[0, -1] / temperature   # temperature controls randomness
        probs  = torch.softmax(last_logits, dim=0)
        next_c = torch.multinomial(probs, 1).item()
        generated += id2char[next_c]

    return generated

print('Generated text:')
print(generate(rnn_model, seed_text='to be or ', n_chars=150))

## 6. The Vanishing Gradient Problem in RNNs

As sequences get longer, gradients must flow through many time steps backward:
```
dL/dh_0 = dL/dh_T * dh_T/dh_{T-1} * ... * dh_1/dh_0
```
Each multiplication by `dh_t/dh_{t-1}` (which involves `tanh` derivative, max 1) shrinks the gradient.
For T=100 steps, gradient at step 0 is essentially 0.

**Result:** RNNs struggle to learn **long-range dependencies**.

The solution: **LSTMs and GRUs** (Notebook 10) — designed specifically to fix this.

---
## YOUR TURN — Mini Project

**Task:** Build a character-level RNN that predicts the next character in a sequence of numbers.

**Steps:**
1. Generate a sequence: repeating pattern like `"0123456789012345678901234..."` (500 chars)
2. Use sliding window of length 10
3. Build a `CharRNN` with hidden_size=64
4. Train for 100 epochs
5. After training, given input `"01234567"` the model should predict `"9"`
6. Test with several starting sequences — does it learn the pattern?

**Challenge:** Try different temperatures in `generate()`:
- `temperature=0.1` (very confident, deterministic)
- `temperature=1.0` (normal sampling)
- `temperature=2.0` (very random)
What do you observe?

In [ ]:
# YOUR CODE HERE
# 1. Generate repeating digit sequence

# 2. Build vocabulary (chars: '0'-'9')

# 3. Create sequences

# 4. Build and train CharRNN

# 5. Test: given '01234567' predict next char

# 6. Explore temperature